# CS336 04: audit the reward before increasing the run

A reward is an instrument with a contract. We will replay retained native responses through the original strict verifier and a separately named mathematical diagnostic, then trace why mixed groups and a task gradient matter. No checkpoint download or GPU job occurs here.

Prerequisites: Chapter 13's response mask, detached group baseline, and frozen reference; Chapter 14's grader coverage. Companion implementation: `src/dongxi_llms/posttraining_assignment.py`, consuming the existing GRPO and mathematical-grading kernels. The archived G4/G8 policies and the new symbolic assignment are separate roots.


## Checkpoint 1: same response, two instruments

### Prediction

Would accepting a boxed answer turn the historical all-zero reward run into a positive-reward experiment? Predict how many retained responses the original verifier must still accept. Then inspect the complete text of every instrument disagreement; unsupported parsing is a separate outcome from an incorrect supported answer.


### Reference solution 1

Run the bounded reference below, then compare it with your prediction.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dongxi_llms').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
from dongxi_llms import posttraining_assignment as pa
from dongxi_llms import reasoning_controls as rc
from dongxi_llms.grpo_lab import group_advantages, clipped_objective, exact_kl
from dongxi_llms.sft_lab import token_loss_sum
torch.set_num_threads(1)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})
audit = pa.native_reward_audit()
print(audit['summary'])
print('Immutable archive sources:', audit['sources'])
for row in audit['rows']:
    if row['disagreement']:
        print({k: row[k] for k in ('archive', 'update', 'source_id', 'expected', 'token_ids', 'original_stop', 'saved_reward', 'richer_status')})
        print(row['raw_response'])
print('One unsupported retained response:', next(r['raw_response'] for r in audit['rows'] if r['richer_status'] == 'UNSUPPORTED'))


### Reference explanation 1

The reference replays all192 sampled responses in the two retained16-update pilots. Each original reward is reproduced from the original stop plus `verify_integer`; any mismatch raises an error. The current archive has zero strict positives, one richer complete positive, and191 unsupported richer parses. The accepted boxed9 response is fully printed, including its original IDs and EOS stop.

The richer diagnostic has a different extraction/format contract. It does not change the rewards that produced the old updates, and it does not prove that a reward-bearing group occurred during the original campaign. Raw source hashes make this a replay of known text, not a new generation result. **Controlled change:** inspect the same accepted text with the stricter `single_integer` format policy in the next checkpoint; answer correctness and format can then disagree.


## Checkpoint 2: adversarial probes and completion

### Prediction

For a correct integer cut off at the cap, should answer correctness, natural termination, and reward all be identical? Predict the rows for prose, a boxed answer, two answers, a wrong answer, an unsafe expression, and a Unicode digit. Change the format policy on the prose probe without changing its arithmetic reference.


### Reference solution 2

Run the bounded reference below, then compare it with your prediction.


In [ ]:
probes = audit['probes']
columns = ['strict_terminated_integer_reward', 'richer_answer_correct', 'richer_format_valid', 'richer_natural_termination', 'richer_complete_reward']
values = [[int(row[key]) for key in columns] for row in probes]
fig, ax = plt.subplots(figsize=(11, 5))
image = ax.imshow(values, vmin=0, vmax=1, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(columns)), ['Strict reward', 'Answer correct', 'Format valid', 'Natural stop', 'Richer complete'])
ax.set_yticks(range(len(probes)), [p['purpose'] for p in probes])
for i,row in enumerate(values):
    for j,value in enumerate(row): ax.text(j,i,str(value),ha='center',va='center')
ax.set_title('Authored adversarial probes: separate grading dimensions')
ax.grid(False)
plt.tight_layout(); plt.show()
for row in probes:
    print(row['purpose'], row['richer_status'], row['richer_reason'], row['raw_response'])
from dongxi_llms.reasoning_evaluation import grade_response
changed = grade_response({'kind': 'math', 'reference': '5', 'extraction': 'boxed_or_final', 'format_policy': 'single_integer'},
                         {'raw_response': 'Final answer: 5', 'stop_reason': 'eos', 'error': None})
print('Same correct prose, stricter format:', {k:changed[k] for k in ('correct','format_valid','task_success','natural_termination')})


### Reference explanation 2

Read across each row before comparing instruments. The capped `5` is correct and formatted, but has no natural stop and earns neither complete reward. Changing prose to `single_integer` leaves its mathematical answer correct while making its format invalid. An unsupported expression receives no correctness credit; no expression is executed. These probes have the independent reference2+3=5 and are authored checker cases, not generated quality evidence.

The richer audit currently permits any format, so `format_valid=True` says the format policy is unconstrained; it says nothing about parser support. Every training reward must name its own extraction, formatting, stopping, and error policy. This notebook's richer protocol is diagnostic only.


![Reference plot: Reference solution 2. Axis labels identify the mechanism quantities.](../../figures/chapter-13/cs336-04_reward_signal_audit-01.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 3: enumerate the random normalizer

### Prediction

For Bernoulli success probability p, the true logit gradient is p(1−p). Predict whether subtracting the inclusive group mean preserves that expectation for group size2, and whether dividing by the sampled standard deviation can be replaced by its expected value. Compare finite groups to the large-group limit. Change p from.5 to.1 while retaining all outcomes.


### Reference solution 3

Run the bounded reference below, then compare it with your prediction.


In [ ]:
finite = pa.finite_group_audit()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for p in (.1, .5, .9):
    for kind,style in [('center-only','--'), ('population-std','-')]:
        rows = [r for r in finite['rows'] if r['success_probability'] == p and r['normalization'] == kind]
        label = f'p={p}, {kind}'
        axes[0].plot([r['group_size'] for r in rows], [r['exact_expected_logit_gradient'] for r in rows], style, marker='o', label=label)
        axes[1].plot([r['group_size'] for r in rows], [r['exact_variance'] for r in rows], style, marker='o', label=label)
axes[0].set(xlabel='Responses in group G', ylabel='Expected ascent gradient dJ/dθ', title='Exact joint enumeration')
axes[1].set(xlabel='Responses in group G', ylabel='Variance of ascent estimator', title='Finite-group variability')
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for r in finite['rows']:
    print({k:r[k] for k in ('success_probability','group_size','normalization','exact_expected_logit_gradient','true_unbaselined_logit_gradient','large_group_std_approximation')})


### Reference explanation 3

The implementation enumerates all2^G joint reward sequences and weights them by their Bernoulli probabilities. For center-only advantages, the exact expectation is ((G−1)/G)p(1−p): the response participates in its own baseline. For population-std normalization, that response also participates in the denominator. At G=2, only mixed pairs contribute; their population std is1/2, giving expectation p(1−p)/(1+2·10⁻⁸).

The std-normalized large-group limit is √(p(1−p)), while the center-only limit is p(1−p). These are differently scaled directions. The plotted variance is that of the complete random estimator, not a sum of independent normalized response variances. No universal variance reduction is asserted. The exact finite calculation and epsilon are part of the protocol.


![Reference plot: Reference solution 3. Axis labels identify the mechanism quantities.](../../figures/chapter-13/cs336-04_reward_signal_audit-02.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 4: difficulty selection spends work and changes population

### Prediction

Suppose three prompt slices have success probabilities.1,.5,.9 and are sampled equally. Predict their distribution after keeping only mixed groups. Will doubling G change both discarded work and the distribution? Compare the conditional gradient to the original expected gradient instead of counting only accepted groups.


### Reference solution 4

Run the bounded reference below, then compare it with your prediction.


In [ ]:
populations = finite['selection_populations']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for pop in populations:
    axes[0].plot(pop['success_probabilities'], pop['mixed_selected_prompt_probabilities'], marker='o', label=f"G={pop['group_size']}")
    axes[1].bar(str(pop['group_size']), pop['attempts_per_selected_group'])
axes[0].axhline(1/3,color='black',ls='--',label='Original uniform prompts')
axes[0].set(xlabel='Prompt success probability p', ylabel='Probability among mixed-selected prompts', title='Selection changes the population')
axes[1].set(xlabel='Group size G',ylabel='Attempted groups / retained group',title='All attempts count as collection work')
axes[0].legend(); plt.tight_layout(); plt.show()
print(populations)
print('Conditional vs original, p=.1/G=2:', [r for r in finite['rows'] if r['success_probability']==.1 and r['group_size']==2])


### Reference explanation 4

A mixed group has probability1−p^G−(1−p)^G. Conditioning on mixed groups reweights originally uniform prompts by this probability, favoring the middle slice for small G. The reference retains the discarded mass and the expected attempts per retained group. It also reports the expected gradient conditional on selection; this is a different estimand from the original population expectation.

The integration assignment chooses eligibility once from the frozen parent's **training** complete-path probabilities, using the predeclared.05–.95 range. Development is only a signal audit and held-out results never choose difficulty. All eligible groups enter the actual update. Its mixed-only population/cost calculation is explicitly counterfactual; it never silently discards all-wrong or all-right collections.


![Reference plot: Reference solution 4. Axis labels identify the mechanism quantities.](../../figures/chapter-13/cs336-04_reward_signal_audit-03.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.


## Checkpoint 5: a nonzero gradient can contain zero reward information

### Prediction

Give every sampled response zero reward, but perturb the policy away from its frozen reference. Predict task-advantage, task-gradient, and KL-gradient norms separately. Then replace the constant rewards with the actual strict sampled path rewards on the same fixed collection. Do not infer task signal from the total norm.


### Reference solution 5

Run the bounded reference below, then compare it with your prediction.


In [ ]:
items, protocol = pa.load_fixture()
train = [i for i in items if i['split']=='train'][:2]
model = rc.make_decoder(33671).double().eval()
reference = rc.frozen_copy(model)
with torch.no_grad(): model.lm_head.weight[7].add_(.15)
repeated = [i for i in train for _ in range(8)]
prompts = pa.prompt_ids(repeated)
tokens, mask = rc.generate(model, prompts, generator=torch.Generator().manual_seed(23))
old = rc.path_log_probabilities(model,prompts,tokens).detach()
current = rc.path_log_probabilities(model,prompts,tokens)
first, later = rc.path_logits(model,prompts,tokens)
with torch.no_grad(): ref_first,ref_later = rc.path_logits(reference,prompts,tokens)
kl = .02*torch.stack((exact_kl(first,ref_first),exact_kl(later,ref_later)),-1).mean()
zero_advantages = group_advantages(torch.zeros(len(train),8,dtype=torch.float64)).flatten()
task = clipped_objective(current,old,zero_advantages,mask)
def gradient_vector(loss):
    return torch.cat([g.reshape(-1) for g in torch.autograd.grad(loss,model.parameters(),retain_graph=True)])
task_grad,kl_grad = gradient_vector(task),gradient_vector(kl)
actual_rewards = torch.tensor([pa.path_grade(t,m,i)['strict_reward'] for t,m,i in zip(tokens,mask,repeated)],dtype=torch.float64).reshape(len(train),8)
actual_advantages = group_advantages(actual_rewards).flatten()
changed_task = clipped_objective(current,old,actual_advantages,mask)
print('Constant reward:',{'advantage_norm':float(zero_advantages.norm()),'task_gradient':float(task_grad.norm()),'weighted_KL_gradient':float(kl_grad.norm()),'total_gradient':float((task_grad+kl_grad).norm())})
print('Changed to actual sampled rewards:',actual_rewards.tolist(),'task-gradient norm:',float(gradient_vector(changed_task).norm()))
fig,ax=plt.subplots()
ax.bar(['Task: constant rewards','Weighted KL','Combined'],[float(task_grad.norm()),float(kl_grad.norm()),float((task_grad+kl_grad).norm())])
ax.set(ylabel='Gradient L2 norm',title='Authored broken-signal control: regularizer motion without task signal')
plt.tight_layout();plt.show()


### Reference explanation 5

All-zero groups produce zero advantages and an exactly zero task gradient. The changed policy still receives a KL gradient pulling it toward the frozen reference, so a nonzero total gradient is insufficient. The changed-input reference computes strict rewards from the sampled IDs and actual EOS, then recomputes the task gradient on that same collection. It reports the actual result even if a group is constant.

This is a fixed CPU gradient probe with no optimizer step and no generated English capability claim. The integration companion records both components for every real update, plus their dot product and the actual objective gradient. The accepted-response SFT arm uses full-vocabulary CE; its printed GRPO gradient is a labeled counterfactual diagnostic, not its training objective.

Continue with [the compatible chain](07_posttraining_integration.ipynb) and the [worked solution](../../../book/solutions/cs336-posttraining-assignment.md).


![Reference plot: Reference solution 5. Axis labels identify the mechanism quantities.](../../figures/chapter-13/cs336-04_reward_signal_audit-04.png)

Read this preview alongside the explanation above. Regenerate it by running the adjacent reference cell; simulated curves and retained measurements keep their stated evidence labels.
